In [ ]:
# 06_explain_shap.ipynb
# Post-hoc Explainability using SHAP

import sys
sys.path.append("../src")
from utils import DATASETS, TARGET, RANDOM_STATE, N_INSTANCES


import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import shap
from sklearn.model_selection import StratifiedShuffleSplit

processed_path = Path("../datasets/processed")
models_path = Path("../results/black_box_models")
results_path = Path("../results/SHAP")
results_path.mkdir(parents=True, exist_ok=True)

CLASS_IDX = 1

for ds in DATASETS:
    print(f"\nDATASET: {ds}")

    data = pd.read_csv(processed_path / f"{ds}_processed.csv")
    X = data.drop(columns=[TARGET])
    y = data[TARGET]

    s = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
    train_idx, test_idx = next(s.split(X, y))
    X_train = X.iloc[train_idx].reset_index(drop=True)
    X_test  = X.iloc[test_idx].reset_index(drop=True)
    y_test  = y.iloc[test_idx].reset_index(drop=True)

    defect_idx = np.where(y_test.values == 1)[0]
    if len(defect_idx) == 0:
        print(f"  No defective instances in {ds}")
        continue
    selected_idx = defect_idx[:N_INSTANCES]

    dataset_model_path = models_path / ds

    for model_dir in sorted(dataset_model_path.iterdir()):
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        print(f" → {model_name}")

        model = joblib.load(model_dir / "model.joblib")

        save_path = results_path / ds / model_name
        save_path.mkdir(parents=True, exist_ok=True)

        is_tree = any(x in model_name.lower() for x in ["random", "gradient", "forest", "boost"])

        if is_tree:
            explainer = shap.TreeExplainer(model)
        else:
            background = X_train.sample(
                n=min(100, len(X_train)),
                random_state=RANDOM_STATE
            ).values
            explainer = shap.KernelExplainer(
                lambda x: model.predict_proba(x),
                background
            )

        records = []

        for i, idx in enumerate(selected_idx):
            instance = X_test.iloc[idx:idx + 1].values

            shap_values = explainer.shap_values(instance)

            if isinstance(shap_values, list):
                shap_values = shap_values[CLASS_IDX]

            shap_vector = shap_values[0].tolist()